# Gen-2 type matchups versus our compact baseline

Compare four logistic regressions with fixed C=.01 and the same five grouped development folds. Each fold reselects the old 300 inputs with RFE; new matchup inputs are appended separately, so the baseline subset stays unchanged. No holdout scoring. Default: review saved results; enable the flag to rerun all comparisons.

Active group: 10 potential/observed-attack effectiveness and knowledge flags (one constant dropped). Team group: 16 offensive coverage, resistance/immunity, HP-weighted advantage and information-coverage measures.


In [ ]:
from pathlib import Path
import sys, csv, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/pokemon_challenge").is_dir())
sys.path.insert(0, str(ROOT / "notebooks/leonard"))
RUN_NEW_EXPERIMENT = False
RUN_FOLDER = ROOT / "outputs/leonard/20261102_154044_358954_matchups"
if RUN_NEW_EXPERIMENT:
    from run_matchup_experiment import run_matchups
    RUN_FOLDER = run_matchups()
print("Artifacts:", RUN_FOLDER)


## Feature construction

Water → Fire = 2; Fire → Water = .5. Multiply against both defensive types; immunity = 0. Omitted valid pairs in the sparse chart mean 1. Species types come from training-fold metadata plus the current battle’s supplied metadata. Unseen enemies get no invented attributes. Unknown matchup values stay missing, with coverage flags and training-fold median imputation.

Type-based potential and actual observed move types are separate. Hidden Power's true type is not assumed; zero-power/fixed-damage moves are excluded. This is not an exact damage calculator. Team pairs include own alive roster versus revealed alive enemies; HP weights use their last observed HP product. Own never-seen members start healthy.


In [ ]:
with (RUN_FOLDER / "cv_results.csv").open() as stream:
    rows = list(csv.DictReader(stream))
for row in rows:
    print(f"{row['candidate']:27s} {float(row['mean_selected_features']):.0f} inputs  {100*float(row['cv_accuracy']):.3f}%")
with (RUN_FOLDER / "paired_comparison.csv").open() as stream:
    for row in csv.DictReader(stream):
        print(row["candidate"], "delta pp:", round(float(row["delta_pp"]), 3), "folds better:", row["folds_better"])


## Decision

Use **RFE300 + team coverage: 316 inputs, 67.259% mean CV**, versus 67.159% for RFE300. Three folds better, one equal, one worse; net eight more correct predictions among 7,996. Active-only features hurt; adding both groups produces only a tiny mean gain.

This is promising but small exploratory evidence, not independent confirmation or a promised higher Kaggle score. The chosen procedure is refitted on all 10,000 labels and generates a new 5,000-row submission. The 300 old selected inputs remain identical to the compact baseline, including in the final full-data fit.


In [ ]:
with (RUN_FOLDER / "new_feature_definitions.csv").open() as stream:
    for row in csv.DictReader(stream):
        print(row["feature"] + ": " + row["description"])
print(json.dumps(json.loads((RUN_FOLDER / "validation_report.json").read_text()), indent=2))
print("Submission:", RUN_FOLDER / "submission.csv")
